# 01. EDA: 결측, 중복, 나이 이상치, 날짜 범위

**목적**: 본 분석(코호트·이탈 특징·세그먼트)에 들어가기 전에 데이터 품질을 점검하고, 이상치·결측을 어떻게 처리할지 근거와 함께 정한다.

- 데이터: `data/kkbox.duckdb` (사용자 1% 샘플)
- 점검 항목: 테이블 구조, 결측, 중복, 나이 이상치, 날짜 범위와 논리 오류, 가입 초기 행동 지표의 유효성
- 이 노트북은 **점검과 처리 방침 결정**까지만 한다. 처리(제외·보정)가 필요하면 근거를 적고, 이후 분석 SQL/노트북에 반영한다.
- 실행 위치: `notebooks/` 폴더 (DB 경로가 `../data/kkbox.duckdb`)

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.rcParams["font.family"] = "Malgun Gothic"   # Windows 한글 폰트
plt.rcParams["axes.unicode_minus"] = False

DB_PATH = "../data/kkbox.duckdb"
con = duckdb.connect(DB_PATH, read_only=True)


def q(sql):
    return con.sql(sql).df()

## 1. 테이블 구조와 규모

In [ ]:
tables = ["members", "transactions_sample", "user_logs_sample", "user_master", "sub_sequence"]
for t in tables:
    n = q(f"SELECT COUNT(*) AS n FROM {t}").iloc[0, 0]
    print(f"[{t}] {n:,}행")
    print(q(f"DESCRIBE {t}")[["column_name", "column_type"]].to_string(index=False))
    print()

## 2. 결측

### 2-1. 테이블별 결측

In [ ]:
def null_report(table):
    cols = q(f"DESCRIBE {table}")["column_name"].tolist()
    sel = ", ".join(f'COUNT(*) - COUNT("{c}") AS "{c}"' for c in cols)
    r = q(f"SELECT COUNT(*) AS _n, {sel} FROM {table}")
    n = int(r.pop("_n").iloc[0])
    out = r.T.rename(columns={0: "결측수"})
    out["결측률(%)"] = (out["결측수"] / n * 100).round(2)
    return out


for t in ["members", "transactions_sample", "user_logs_sample", "user_master"]:
    print(f"[{t}]")
    print(null_report(t).to_string())
    print()

### 2-2. user_master의 가입 초기 행동 지표 결측
`has_early_window`는 "청취 로그가 있다"가 아니라 **"로그 수집 시작일(2015-01-01) 이후에 가입했다"**는 뜻이다 (`02_build_user_master.sql`). 따라서 FALSE는 관측 시작 전 가입자로, 가입 후 30일 로그를 볼 수 없다. 이 집단의 활동일 0은 실제 0이 아니라 **관측 불가**이므로 초기 행동 분석에서 쓰지 않는다.

TRUE 집단 안의 결측은 두 가지로 나뉜다.
- 로그 행이 아예 없는 사용자: 지표 전체가 NULL
- 로그는 있지만 가입 후 30일 안에는 없는 사용자: 활동일은 0, 청취 시간·곡 수·스킵 비율은 NULL

In [ ]:
um = q("SELECT * FROM user_master")
early_cols = ["active_days_7d", "active_days_30d", "listen_secs_30d", "unique_songs_30d", "skip_ratio_30d"]

print(um["has_early_window"].value_counts(dropna=False).to_string())
print()
print("has_early_window별 결측률(%)")
print(um.groupby("has_early_window")[early_cols].apply(lambda d: d.isna().mean() * 100).round(2).to_string())

### 2-3. TRUE 집단의 결측 원인 분해
위 설명이 맞다면 (로그 행 없음) + (30일 활동 0) = 청취 시간 결측이어야 한다.

In [ ]:
t = um[um["has_early_window"] == True]
no_log = int(t["active_days_30d"].isna().sum())
zero_active = int((t["active_days_30d"] == 0).sum())
listen_null = int(t["listen_secs_30d"].isna().sum())
print(f"로그 행 없음: {no_log:,}명")
print(f"로그는 있으나 가입 후 30일 활동 0: {zero_active:,}명")
print(f"청취 시간 결측: {listen_null:,}명  (= {no_log:,} + {zero_active:,} = {no_log + zero_active:,})")

- 결측의 대부분은 **가입 초기에 활동이 없는 사용자**다. 이들은 가설 H1(초기 활동이 적을수록 이탈)의 핵심 집단이므로 **제외하지 않고** 활동일·청취 시간·곡 수를 0으로 두고 "초기 활동 없음" 표시를 붙여 쓴다. 스킵 비율은 재생이 없으면 정의되지 않으므로 NULL로 두고 결측 표시를 쓴다.

## 3. 중복

In [ ]:
rows = []
for t in ["members", "transactions_sample", "user_logs_sample"]:
    r = q(f"""
        SELECT COUNT(*) AS 전체행,
               (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM {t})) AS 고유행
        FROM {t}
    """).iloc[0]
    rows.append({"table": t, "전체행": int(r["전체행"]), "고유행": int(r["고유행"]),
                 "완전중복행": int(r["전체행"] - r["고유행"])})
print(pd.DataFrame(rows).to_string(index=False))

print()
print("사용자 키(msno) 중복")
for t in ["members", "user_master"]:
    r = q(f"SELECT COUNT(*) AS 행수, COUNT(DISTINCT msno) AS 고유msno FROM {t}").iloc[0]
    print(f"  {t}: 행 {int(r['행수']):,} / 고유 msno {int(r['고유msno']):,}")

- 거래(`transactions_sample`)의 완전 중복행은 같은 날 같은 조건으로 두 번 결제된 정상 거래일 수도 있어, 건수만 보고 바로 지우지 않는다. 중복이 있으면 취소 거래 여부와 금액을 같이 보고 처리 방침을 정한다.

## 4. 나이 이상치
`user_master.age`의 분포를 확인한다. 원본 나이 컬럼은 0이나 음수, 100 초과처럼 비현실적인 값이 섞여 있을 수 있다.

In [ ]:
print(um["age"].describe().round(2).to_string())
print()
bucket = q("""
    SELECT CASE WHEN age IS NULL THEN '0_NULL'
                WHEN age <= 0 THEN '1_<=0'
                WHEN age < 10 THEN '2_1-9'
                WHEN age <= 80 THEN '3_10-80'
                ELSE '4_>80' END AS 구간,
           COUNT(*) AS n
    FROM user_master
    GROUP BY 1 ORDER BY 1
""")
bucket["비율(%)"] = (bucket["n"] / bucket["n"].sum() * 100).round(2)
print(bucket.to_string(index=False))
print()
print("가장 많이 나온 나이 값 (placeholder 의심)")
print(um["age"].value_counts(dropna=False).head(10).to_string())

In [ ]:
valid = um.loc[(um["age"] >= 10) & (um["age"] <= 80), "age"]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(valid, bins=range(10, 82, 2), color="#4C72B0")
ax.set_title("나이 분포 (10~80세만)")
ax.set_xlabel("나이")
ax.set_ylabel("사용자 수")
plt.tight_layout()
plt.show()

## 5. 날짜 범위와 논리 오류

### 5-1. 날짜형 컬럼의 범위
날짜·시간으로 보이는 컬럼(타입 또는 이름 기준)을 자동으로 찾아 최소·최대·결측 수를 본다.

In [ ]:
date_like = []
for t in tables:
    d = q(f"DESCRIBE {t}")
    for _, r in d.iterrows():
        name, typ = r["column_name"], r["column_type"]
        if typ in ("DATE", "TIMESTAMP") or any(k in name.lower() for k in ["date", "time", "expire"]):
            date_like.append((t, name, typ))

rows = []
for t, c, typ in date_like:
    r = q(f'SELECT MIN("{c}") AS mn, MAX("{c}") AS mx, COUNT(*) - COUNT("{c}") AS nulls FROM {t}').iloc[0]
    rows.append({"table": t, "column": c, "type": typ, "min": r["mn"], "max": r["mx"], "결측": int(r["nulls"])})
pd.DataFrame(rows).to_string(index=False)

### 5-2. 날짜 논리 점검 (user_master)

In [ ]:
q("""
    SELECT COUNT(*) AS 전체,
           SUM(CASE WHEN registration_date IS NULL THEN 1 ELSE 0 END) AS 가입일_없음,
           SUM(CASE WHEN first_tx_date < registration_date THEN 1 ELSE 0 END) AS 첫결제가_가입일보다_이름,
           SUM(CASE WHEN cohort_month <> date_trunc('month', registration_date) THEN 1 ELSE 0 END) AS 코호트월_불일치
    FROM user_master
""").to_string(index=False)

- `첫결제가_가입일보다_이름`은 1명이다. 가입 후 N일을 계산하면 음수가 되므로 시간 기반 분석에서만 제외한다.

### 5-3. 멤버십 만료일 이상값
`membership_expire_date`에 관측 기간 밖 값(1970-01-01, 2025-05-19 등)이 있다. 이탈 판정(`01`)은 만료일의 누적 최댓값(`covered_until`)을 쓰므로, 이상값이 판정을 바꾸는지 본다.

In [ ]:
q("""
    SELECT is_cancel, COUNT(*) AS n,
           SUM(CASE WHEN YEAR(membership_expire_date) < 2015 THEN 1 ELSE 0 END) AS expire_lt_2015,
           SUM(CASE WHEN YEAR(membership_expire_date) > 2018 THEN 1 ELSE 0 END) AS expire_gt_2018,
           SUM(CASE WHEN membership_expire_date < transaction_date THEN 1 ELSE 0 END) AS expire_before_tx
    FROM transactions_sample
    GROUP BY is_cancel ORDER BY is_cancel
""").to_string(index=False)

In [ ]:
q("""
    SELECT COUNT(DISTINCT s.msno) AS 영향_사용자,
           COUNT(DISTINCT CASE WHEN u.is_censored_30 = 1 THEN s.msno END) AS 판정불가_사용자,
           COUNT(DISTINCT CASE WHEN u.churned_30 = 1 THEN s.msno END) AS 최종이탈_사용자
    FROM sub_sequence s JOIN user_master u USING (msno)
    WHERE YEAR(s.membership_expire_date) < 2015
       OR YEAR(s.membership_expire_date) > 2018
       OR s.membership_expire_date < s.transaction_date
""").to_string(index=False)

- 취소 거래(`is_cancel=1`)의 만료일이 결제일보다 앞서는 것은 취소 거래의 특성이라 문제가 아니며, `sub_sequence`(비취소 196,284행)에서 이미 제외된다.
- 비취소 거래의 이상값은 최대 약 149행(0.08%), 영향 사용자는 40명(분석 대상 19,838명의 약 0.2%)이다. 이 수는 **상한**이다. 영향 사용자 중에는 이상값과 무관하게 판정 불가이거나 이탈인 사용자가 섞여 있다.
- 만료일이 먼 미래인 경우 그 사용자는 계속 구독 중으로 계산되어 이탈이 아닌 판정 불가로 분류될 수 있고, 만료일이 1970년인 첫 거래는 이탈 후 복귀로 잘못 기록될 수 있다.

## 6. 가입 초기 행동 지표의 유효성
`has_early_window = TRUE`인 사용자만 대상으로, 물리적으로 불가능한 값이 있는지 확인한다.

In [ ]:
early = um[um["has_early_window"] == True]
print(f"대상: {len(early):,}명")
print()
print(early[early_cols].describe().round(2).to_string())
print()
checks = {
    "active_days_7d > 7": (early["active_days_7d"] > 7).sum(),
    "active_days_30d > 30": (early["active_days_30d"] > 30).sum(),
    "active_days_7d > active_days_30d": (early["active_days_7d"] > early["active_days_30d"]).sum(),
    "listen_secs_30d < 0": (early["listen_secs_30d"] < 0).sum(),
    "listen_secs_30d > 30일 전체(2,592,000초)": (early["listen_secs_30d"] > 30 * 86400).sum(),
    "skip_ratio_30d < 0 또는 > 1": ((early["skip_ratio_30d"] < 0) | (early["skip_ratio_30d"] > 1)).sum(),
}
print(pd.Series(checks, name="건수").to_string())

### 6-2. 일 단위 청취 시간 점검
위 점검은 30일 합계 기준이라 하루 값의 이상을 놓친다. 원본 일 단위 `total_secs`를 직접 본다.

In [ ]:
q("""
    SELECT COUNT(*) AS 전체행,
           SUM(CASE WHEN total_secs < 0 THEN 1 ELSE 0 END) AS 음수,
           SUM(CASE WHEN total_secs > 86400 THEN 1 ELSE 0 END) AS 하루_초과
    FROM user_logs_sample
""").to_string(index=False)

- 하루 초과(1,482행)는 `02`에서 이미 86,400초로 상한 처리되어 있다.
- 음수(582행)는 상한만 있고 하한이 없어서 6명의 30일 합계가 음수로 나왔다. `02`에서 음수 행을 합산에서 제외하도록 고쳤다 (재실행 후 6-1의 `listen_secs_30d < 0`이 0이 되는지 확인).

## 7. 범주형 변수 분포

In [ ]:
for col in ["first_plan_days", "first_payment_method", "first_auto_renew", "registered_via", "gender", "city"]:
    vc = um[col].value_counts(dropna=False).head(10)
    out = pd.DataFrame({"n": vc, "비율(%)": (vc / len(um) * 100).round(2)})
    print(f"[{col}] 상위 {len(out)}개 (고유값 {um[col].nunique(dropna=False)}개)")
    print(out.to_string())
    print()

## 8. 분석 대상 확인
`user_master`에는 있지만 `sub_sequence`에는 없는 사용자(취소 거래만 있는 사용자)는 퍼널·이탈 분석에서 제외된다. EDA 결과를 읽을 때 이 집단을 구분해서 본다.

In [ ]:
in_scope = ~um["msno"].isin(q("SELECT msno FROM sub_sequence")["msno"])
print(f"user_master 전체: {len(um):,}명")
print(f"sub_sequence에 없음(분석 제외): {int(in_scope.sum()):,}명")
print(f"분석 대상: {int((~in_scope).sum()):,}명")

## 9. 점검 결과와 처리 방침

| 점검 항목 | 결과 | 처리 방침과 근거 |
|---|---|---|
| 결측 (user_master) | `age` 10,072명(50.52%), `gender` 9,936명(49.84%). `city`, `registered_via`, `first_*`는 0. `churned_*` 계열 99명(0.50%)은 분석 제외 대상 99명과 같음 | `gender`는 "미입력"을 별도 범주로 둔다. `age`는 회귀에서 결측 표시 변수를 함께 쓰고, 나이 보유자만으로 민감도 분석을 한다. 절반을 지우면 표본이 크게 줄고, 프로필 미입력 자체가 행동 신호일 수 있다. 99명은 분석 제외 |
| 가입 초기 행동 지표 결측 | `has_early_window`=TRUE 9,104명 중 로그 행 없음 151명, 가입 후 30일 활동 0인 사용자 876명 (청취 시간 결측 1,027명 = 151 + 876). FALSE 10,833명은 2015-01-01 이전 가입자로 관측 불가 | 초기 행동 분석은 TRUE 집단만 쓴다. 결측은 대부분 "초기 활동 없음"이고 H1의 핵심 집단이라 제외하지 않고, 활동일·청취 시간·곡 수는 0으로 두고 "초기 활동 없음" 표시를 붙인다. 스킵 비율은 재생이 없어 정의되지 않으므로 NULL로 두고 결측 표시를 쓴다 |
| 완전 중복행 (거래·로그·회원) | 세 테이블 모두 0건, `msno` 중복 0 | 조치 없음. 완전히 같은 행만 본 것이며 청취 로그의 (`msno`, `log_date`) 단위 중복은 확인하지 않았다 |
| 나이 이상치 | 값이 있는 9,865명은 13~75세(중앙값 28). 10~80 밖의 값은 `02`에서 이미 NULL 처리됨 | 추가 처리 없음. `bd=0`(미입력)과 이상치가 모두 NULL로 섞여 있어 구분할 수 없다는 점을 README에 적는다 |
| 날짜 범위 | 거래·로그 2015-01-01~2017-03-31, 가입일 2004-03-27~2017-03-31. 만료일은 1970-01-01~2025-05-19. 비취소 거래의 만료일 이상값은 최대 약 149행(0.08%), 영향 사용자 40명(약 0.2%, 상한) | 이탈 정의(`01`)는 바꾸지 않고 한계에 기록한다. 영향이 0.2% 이하여서 정의를 고치면 검증된 숫자(공식 라벨 일치도 등)를 다시 만들어야 하는 비용이 더 크다. 코호트 유지율은 2015-01 이후 가입자로 한정한다 |
| 첫 결제가 가입일보다 앞선 사용자 | 1명 (가입일 결측 0, 코호트월 불일치 0) | 가입 후 N일 계산에서 음수가 되므로 시간 기반 분석에서만 제외한다. 19,937명 중 1명이라 영향은 무시할 만하다 |
| 행동 지표의 불가능한 값 | `active_days_*`, `skip_ratio` 범위 위반 0건. `listen_secs_30d` 음수 6명. 일 단위로는 음수 582행, 하루 초과 1,482행 | 하루 초과는 `02`에서 이미 86,400초로 상한 처리되어 있다. 음수 행은 합산에서 제외하도록 `02`를 수정했다 (사용자는 유지). 재실행 후 음수 사용자가 0명인지 확인한다 |